# 01 — Analisis Exploratorio de Datos (EDA)

## Problema de negocio

Este proyecto aborda la prediccion y analisis de salarios de desarrolladores de software mediante tres tareas complementarias:

1. **Regresion:** predecir el valor exacto de `salary_usd` (rango ~12K-281K USD).
2. **Clasificacion binaria:** predecir si un desarrollador supera la mediana salarial del mercado (`salary_above_median`, umbral = mediana del train set).
3. **Clustering (no supervisado):** identificar perfiles de desarrolladores usando features demograficas y profesionales (sin usar salary como input).

### Justificacion

- La **regresion** permite estimar compensacion exacta para benchmarking salarial.
- La **clasificacion binaria** simplifica la decision: "este perfil esta por encima o por debajo del mercado". Al usar la mediana como umbral se garantiza un balance ~50/50 en las clases.
- El **clustering** revela segmentos naturales del mercado (senior en empresa grande vs junior en startup, etc.) sin supervision.

### Nota sobre el preprocesamiento

El pipeline completo de limpieza y transformacion de datos se encuentra documentado en `etl/notebooks/`. Este notebook se enfoca exclusivamente en lo relevante para las decisiones de modelado: distribucion del target, relacion features-target, y cardinalidad de categoricas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Carga del dataset crudo

Cargamos el dataset original para mostrar su estado antes del pipeline ETL y justificar por que es necesario un preprocesamiento previo al modelado.

In [ ]:
raw = pd.read_csv('../etl/data/raw/software_developer_salary_raw.csv')
print(f'Shape: {raw.shape[0]} filas x {raw.shape[1]} columnas')
print(f'\nColumnas: {list(raw.columns)}')
raw.head(10)

In [ ]:
raw.dtypes

## 2. Estado del dataset crudo

Verificacion rapida de problemas de calidad que justifican la existencia del pipeline ETL.

In [ ]:
# Valores nulos por columna
nulls = raw.isnull().sum()
nulls_pct = (nulls / len(raw) * 100).round(1)
null_summary = pd.DataFrame({'nulos': nulls, 'porcentaje': nulls_pct})
print('Valores nulos por columna:')
null_summary[null_summary['nulos'] > 0]

In [ ]:
# Duplicados
n_duplicados = raw.duplicated().sum()
print(f'Filas duplicadas: {n_duplicados} ({n_duplicados/len(raw)*100:.1f}%)')

# Problemas conocidos en el raw
print(f'\nExperiencia negativa: {(raw["experience"] < 0).sum()} filas')
print(f'Education con casing inconsistente: {raw["education"].nunique()} valores unicos')
print(f'  Ejemplos: {sorted(raw["education"].dropna().unique())}')
print(f'\nCountry con casing inconsistente: {raw["country"].nunique()} valores unicos')

In [ ]:
# Valores duplicados dentro de campos multi-valor (frameworks)
frameworks_with_dups = raw['frameworks'].dropna().apply(
    lambda x: len(x.split(', ')) != len(set(x.split(', ')))
)
print(f'Filas con frameworks duplicados internamente: {frameworks_with_dups.sum()}')
print(f'Ejemplo: {raw.loc[frameworks_with_dups[frameworks_with_dups].index[0], "frameworks"]}')

**Conclusion:** el dataset crudo tiene nulos en `education` y `frameworks`, experiencia negativa, casing inconsistente en categoricas, y valores duplicados dentro de campos multi-valor. Todo esto se resuelve en el pipeline ETL (`etl/`) que produce `clean.csv`.

## 3. Carga del dataset limpio

A partir de aqui trabajamos con `clean.csv`, resultado del pipeline ETL. Este es el dataset base para modelado.

In [ ]:
df = pd.read_csv('../etl/data/processed/clean.csv')
print(f'Shape: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'\nColumnas: {list(df.columns)}')
print(f'\nNulos restantes: {df.isnull().sum().sum()}')
df.head()

## 4. Analisis del target: `salary_usd`

La variable objetivo para regresion. Necesitamos entender su distribucion para elegir modelos y metricas adecuadas.

In [ ]:
# Estadisticas descriptivas
stats = df['salary_usd'].describe()
stats['mediana'] = df['salary_usd'].median()
stats['skew'] = df['salary_usd'].skew()
stats['kurtosis'] = df['salary_usd'].kurtosis()
stats['IQR'] = stats['75%'] - stats['25%']
print('Estadisticas de salary_usd:')
stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df['salary_usd'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(df['salary_usd'].median(), color='red', linestyle='--',
                label=f'Mediana: ${df["salary_usd"].median():,.0f}')
axes[0].axvline(df['salary_usd'].mean(), color='orange', linestyle='--',
                label=f'Media: ${df["salary_usd"].mean():,.0f}')
axes[0].set_xlabel('Salario (USD)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribucion de salary_usd')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['salary_usd'], vert=True)
axes[1].set_ylabel('Salario (USD)')
axes[1].set_title('Boxplot de salary_usd')

plt.tight_layout()
plt.show()

**Interpretacion para modelado:**
- Si la distribucion es aproximadamente simetrica, modelos lineales (Ridge, Lasso) son candidatos viables sin transformacion del target.
- Si hay sesgo positivo (skew > 0.5), considerar log-transform del target o modelos robustos a outliers (Random Forest, Gradient Boosting).
- La presencia de outliers visibles en el boxplot sugiere que MAE puede ser mas informativa que MSE como metrica de evaluacion.

### 4.1 Balance de la clase binaria derivada

Para clasificacion, usamos la mediana como umbral para crear `salary_above_median`. Verificamos que el balance sea ~50/50 (propiedad inherente de la mediana).

In [ ]:
mediana = df['salary_usd'].median()
above_median = (df['salary_usd'] > mediana).astype(int)

print(f'Mediana del dataset: ${mediana:,.0f} USD')
print(f'\nBalance de clases (salary_above_median):')
print(f'  Clase 0 (<=mediana): {(above_median == 0).sum()} ({(above_median == 0).mean()*100:.1f}%)')
print(f'  Clase 1 (>mediana):  {(above_median == 1).sum()} ({(above_median == 1).mean()*100:.1f}%)')
print(f'\nRatio: {(above_median == 1).sum() / (above_median == 0).sum():.3f}')
print('\nEl balance ~50/50 confirma que no necesitamos tecnicas de balanceo (SMOTE, undersampling).')

## 5. Relacion features vs target

Exploramos que features tienen mayor poder predictivo sobre `salary_usd`. Esto informa la seleccion de modelos y la importancia esperada de cada variable.

### 5.1 Correlacion entre variables numericas

In [ ]:
numeric_cols = ['experience', 'num_languages', 'num_frameworks', 'salary_usd']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlacion entre variables numericas y salary_usd')
plt.tight_layout()
plt.show()

print('\nCorrelacion con salary_usd:')
print(corr_matrix['salary_usd'].drop('salary_usd').sort_values(ascending=False))

**Interpretacion:** `experience` deberia ser el predictor numerico mas fuerte. `num_languages` y `num_frameworks` aportan informacion sobre versatilidad tecnica pero su correlacion con salario puede ser menor.

### 5.2 Salary por pais (top 10)

In [ ]:
top_countries = df['country'].value_counts().head(10).index
df_top_countries = df[df['country'].isin(top_countries)]

# Ordenar por mediana de salario
order = df_top_countries.groupby('country')['salary_usd'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df_top_countries, x='country', y='salary_usd', order=order, ax=ax)
ax.set_xlabel('Pais')
ax.set_ylabel('Salario (USD)')
ax.set_title('Distribucion de salary_usd por pais (top 10 por frecuencia)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

**Interpretacion para modelado:** si hay diferencias significativas entre paises, `country` sera un feature importante. La alta cardinalidad (muchos paises) implica que OneHotEncoding generara muchas columnas — considerar agrupar paises con pocas observaciones.

### 5.3 Salary por nivel educativo

In [ ]:
order_edu = df.groupby('education')['salary_usd'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='education', y='salary_usd', order=order_edu, ax=ax)
ax.set_xlabel('Nivel educativo')
ax.set_ylabel('Salario (USD)')
ax.set_title('Distribucion de salary_usd por nivel educativo')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

**Interpretacion para modelado:** si las medianas son similares entre niveles educativos, `education` tendra bajo poder predictivo por si sola. Puede interactuar con `experience`.

### 5.4 Salary por tamano de empresa

In [ ]:
# Orden logico por tamano
size_order = ['1-10', '11-50', '51-200', '201-1000', '1001-5000', '5000+']
# Filtrar solo los que existen en el dataset
size_order = [s for s in size_order if s in df['company_size'].values]

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='company_size', y='salary_usd', order=size_order, ax=ax)
ax.set_xlabel('Tamano de empresa')
ax.set_ylabel('Salario (USD)')
ax.set_title('Distribucion de salary_usd por tamano de empresa')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

**Interpretacion para modelado:** `company_size` tiene un orden natural (ordinal). Se puede codificar como OneHot o como variable ordinal numerica. Si la relacion con salario es monotona, OrdinalEncoder podria capturar la tendencia directamente.

### 5.5 Scatter: experiencia vs salary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(df['experience'], df['salary_usd'], alpha=0.3, s=10, color='steelblue')
ax.set_xlabel('Anos de experiencia')
ax.set_ylabel('Salario (USD)')
ax.set_title('Experiencia vs Salario')
plt.tight_layout()
plt.show()

**Interpretacion:** si la relacion es lineal, modelos lineales capturaran bien el efecto de experiencia. Si hay un plateau o relacion no lineal, modelos basados en arboles seran superiores.

## 6. Cardinalidad de variables categoricas

La cardinalidad determina la estrategia de encoding y la dimensionalidad resultante del feature space.

In [ ]:
categoricas = ['country', 'education', 'company_size', 'languages', 'frameworks']

print('Cardinalidad de variables categoricas:')
print('=' * 45)
for col in categoricas:
    print(f'{col:20s} -> {df[col].nunique():4d} valores unicos')

print(f'\n{"="*45}')
print(f'Total de filas: {len(df)}')

In [ ]:
# Distribucion de education y company_size (baja cardinalidad)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['education'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Distribucion de education')
axes[0].set_xlabel('Nivel educativo')
axes[0].set_ylabel('Frecuencia')
axes[0].tick_params(axis='x', rotation=30)

df['company_size'].value_counts().reindex(size_order).plot(kind='bar', ax=axes[1],
                                                           color='steelblue', edgecolor='black')
axes[1].set_title('Distribucion de company_size')
axes[1].set_xlabel('Tamano de empresa')
axes[1].set_ylabel('Frecuencia')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Lenguajes y frameworks mas comunes (extraidos de campos multi-valor)
all_languages = df['languages'].str.split(', ').explode()
all_frameworks = df['frameworks'].str.split(', ').explode()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

all_languages.value_counts().head(15).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 lenguajes de programacion')
axes[0].set_xlabel('Frecuencia')
axes[0].invert_yaxis()

all_frameworks.value_counts().head(15).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Top 15 frameworks')
axes[1].set_xlabel('Frecuencia')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f'Lenguajes unicos: {all_languages.nunique()}')
print(f'Frameworks unicos: {all_frameworks.nunique()}')

**Implicaciones para encoding:**
- `education` y `company_size` tienen baja cardinalidad → OneHotEncoder directo (pocas columnas).
- `country` tiene cardinalidad media-alta → OneHotEncoder es viable si hay suficientes observaciones por pais; alternativamente agrupar paises con pocas muestras en "Other".
- `languages` y `frameworks` son campos multi-valor → requieren tratamiento especial (MultiLabelBinarizer o usar solo `num_languages`/`num_frameworks` como proxy numerico).
- La dimensionalidad total tras encoding sera manejable para modelos lineales y de arboles con el tamano del dataset (~9K filas).

## 7. Conclusiones del EDA para modelado

### Features mas predictivas (hipotesis a validar)

1. **`experience`** — correlacion directa con salario; probablemente el predictor mas fuerte.
2. **`country`** — diferencias geograficas en compensacion son conocidas en la industria tech.
3. **`company_size`** — empresas grandes suelen pagar mas; relacion potencialmente monotona.
4. **`education`** — impacto moderado; posible interaccion con experiencia.
5. **`num_languages` / `num_frameworks`** — proxies de versatilidad tecnica; correlacion debil esperada.

### Transformaciones para los pipelines del notebook 02

| Tipo de variable | Tratamiento en Pipeline |
|---|---|
| Numericas (`experience`, `num_languages`, `num_frameworks`) | StandardScaler |
| Categoricas de baja cardinalidad (`education`, `company_size`) | OneHotEncoder |
| Categorica de alta cardinalidad (`country`) | OneHotEncoder (con `handle_unknown='ignore'`) |
| Multi-valor (`languages`, `frameworks`) | Usar conteo numerico o MultiLabelBinarizer |

### Viabilidad de los 3 problemas planteados

- **Regresion:** viable. El target tiene varianza suficiente y features con correlacion no trivial.
- **Clasificacion binaria:** viable. Balance ~50/50 garantizado por construccion (mediana como umbral). No requiere tecnicas de balanceo.
- **Clustering:** viable. Variables demograficas y profesionales (country, education, company_size, experience) permiten segmentacion sin usar salary como input.

### Siguiente paso

El notebook `02_supervised_modeling.ipynb` implementara los pipelines de sklearn con train/test split, encoding y escalado integrados, y entrenara multiples modelos de regresion y clasificacion.